# OptBench Progress Monitor

Tracks progress of the per-(exp, model, seed) slurm jobs launched by `check_slurm.sh`.

- **Section 1** — done/total per (exp, model, seed), pulling recipe counts directly from `exp1.py`/`exp2.py` and finished runs from wandb.
- **Section 2** — hours-remaining estimates per slurm job, from prior finished runtimes on the same model.

In [1]:
import importlib.util
import re
from pathlib import Path

import pandas as pd
import wandb

# scripts/opt-bench has a '-' in its name so it is not a valid python package;
# load exp1/exp2 by file path instead.
_HERE = Path.cwd() if Path.cwd().name == 'opt-bench' else Path('scripts/opt-bench')

def _load(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

exp1 = _load('exp1', _HERE / 'exp1.py')
exp2 = _load('exp2', _HERE / 'exp2.py')

print(f'exp1 whitebox recipes: {len(exp1.WHITEBOX_OPTIMIZER_CONFIGS)} + 1 nanogcg')
print(f'exp2 variants: {len(exp2._build_variants(n_layers=0))}')
print(f'msg_ids: {len(exp1.MSG_IDS)}   seeds: {exp1.SEEDS}')

exp1 whitebox recipes: 15 + 1 nanogcg
exp2 variants: 7
msg_ids: 10   seeds: [42, 123, 777]


In [2]:
# ─── Job matrix (must match check_slurm.sh / run_all.sh) ──────────────────
LETTER_TO_MODEL = {
    'l': 'meta-llama/Llama-3.1-8B-Instruct',
    'g': 'google/gemma-3-12b-it',
    'q': 'Qwen/Qwen3-8B',
}
MODEL_SHORTS = {letter: m.split('/')[-1] for letter, m in LETTER_TO_MODEL.items()}
SEEDS = list(exp1.SEEDS)
MSG_IDS = list(exp1.MSG_IDS)
EXPS = [1, 2]

WANDB_ENTITY = exp1.WANDB_ENTITY
EXP1_PROJECT = exp1.WANDB_PROJECT           # tropt-optbench
EXP1_BB_PROJECT = exp1.WANDB_PROJECT_BB     # tropt-optbench-bb (OpenAI blackbox)
EXP2_PROJECT = exp2.WANDB_PROJECT           # tropt-enhancebench

BLACKBOX_MODEL = 'openai/gpt-5-nano'
BLACKBOX_MODEL_SHORT = BLACKBOX_MODEL.split('/')[-1]

# Recipe names per experiment
EXP1_RECIPES = [c.name for c in exp1.WHITEBOX_OPTIMIZER_CONFIGS] + ['nanogcg']
EXP1_BB_RECIPES = [c.name for c in exp1.BLACKBOX_OPTIMIZER_CONFIGS]
EXP2_RECIPES = [v.name for v in exp2._build_variants(n_layers=0)]

# Expected runs per slurm job
# Each 'recipe' (optimizer/variant) runs once per msg_id per seed in the outer loop;
# for per-(model, seed) jobs the seed is fixed → expected = n_recipes × n_msgs.
def expected_count(exp_kind: str) -> int:
    if exp_kind == '1':
        return len(EXP1_RECIPES) * len(MSG_IDS)
    if exp_kind == '1bb':
        # exp1chat is a single job covering ALL seeds × ALL msgs × blackbox optimizers
        return len(EXP1_BB_RECIPES) * len(MSG_IDS) * len(SEEDS)
    if exp_kind == '2':
        return len(EXP2_RECIPES) * len(MSG_IDS) + len(EXP2_RECIPES)  # single + multi
    raise ValueError(exp_kind)

print(f'Expected runs per exp1 job:   {expected_count("1")}')
print(f'Expected runs per exp1chat:   {expected_count("1bb")}')
print(f'Expected runs per exp2 job:   {expected_count("2")}')


Expected runs per exp1 job:   160
Expected runs per exp1chat:   210
Expected runs per exp2 job:   77


In [3]:
# ─── Fetch finished runs from wandb ──────────────────────────────────────
api = wandb.Api()

def fetch_runs(project: str, state: str | None = None):
    filters = {'state': state} if state else None
    try:
        return list(api.runs(f'{WANDB_ENTITY}/{project}', filters=filters))
    except Exception as e:
        print(f'  [warn] {project}: {e}')
        return []

exp1_runs = fetch_runs(EXP1_PROJECT, state='finished')
exp1_bb_runs = fetch_runs(EXP1_BB_PROJECT, state='finished')
exp2_runs = fetch_runs(EXP2_PROJECT, state='finished')
print(f'finished runs — exp1: {len(exp1_runs)}  exp1_bb: {len(exp1_bb_runs)}  exp2: {len(exp2_runs)}')

# Patterns for parsing the run_name fields.
# exp1 whitebox/nanogcg AND exp1 blackbox share the same prefix 'optbench['.
RE_EXP1 = re.compile(r'^optbench\[(?P<recipe>[^,]+),(?P<model>[^,]+),m=(?P<msg>\d+),s=(?P<seed>\d+)\]$')
RE_EXP2_SINGLE = re.compile(r'^enhancebench\[(?P<recipe>[^,]+),(?P<model>[^,]+),m=(?P<msg>\d+),s=(?P<seed>\d+)\]$')
RE_EXP2_MULTI = re.compile(r'^enhancebench_multi\[(?P<recipe>[^,]+),(?P<model>[^,]+),s=(?P<seed>\d+)\]$')

def parse_run(r, source_project: str):
    for kind, regex in (('exp1', RE_EXP1), ('exp2', RE_EXP2_SINGLE), ('exp2', RE_EXP2_MULTI)):
        m = regex.match(r.name or '')
        if m:
            d = m.groupdict()
            d['run'] = r
            # Disambiguate exp1 whitebox vs blackbox by the source project.
            if kind == 'exp1':
                d['exp'] = '1bb' if source_project == EXP1_BB_PROJECT else '1'
            else:
                d['exp'] = '2'
            return d
    return None

parsed = []
for r in exp1_runs:
    p = parse_run(r, EXP1_PROJECT)
    if p: parsed.append(p)
for r in exp1_bb_runs:
    p = parse_run(r, EXP1_BB_PROJECT)
    if p: parsed.append(p)
for r in exp2_runs:
    p = parse_run(r, EXP2_PROJECT)
    if p: parsed.append(p)
print(f'parsed: {len(parsed)}')


wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from C:\Users\mtnbt\.netrc.


wandb: WARNING A graphql request initiated by the public wandb API timed out (timeout=19 sec). Create a new API with an integer timeout larger than 19, e.g., `api = wandb.Api(timeout=29)` to increase the graphql timeout.


finished runs — exp1: 281  exp1_bb: 2  exp2: 14
parsed: 297


In [4]:
# ─── Progress table: done / total per slurm job ──────────────────────────
rows = []

# exp1 + exp2: per (model, seed)
for exp in ['1', '2']:
    total = expected_count(exp)
    for letter, model_full in LETTER_TO_MODEL.items():
        short = MODEL_SHORTS[letter]
        for seed_idx, seed in enumerate(SEEDS, start=1):
            done = sum(
                1 for p in parsed
                if p['exp'] == exp and p['model'] == short and int(p['seed']) == seed
            )
            rows.append({
                'job': f'exp{exp}{letter}{seed_idx}',
                'exp': exp,
                'model': short,
                'seed': seed,
                'done': done,
                'total': total,
                'pct': f'{100*done/total:5.1f}%',
            })

# exp1chat: the single OpenAI blackbox job — aggregates over all seeds and msgs.
bb_total = expected_count('1bb')
bb_done = sum(
    1 for p in parsed
    if p['exp'] == '1bb' and p['model'] == BLACKBOX_MODEL_SHORT
)
rows.append({
    'job': 'exp1chat',
    'exp': '1bb',
    'model': BLACKBOX_MODEL_SHORT,
    'seed': 'all',
    'done': bb_done,
    'total': bb_total,
    'pct': f'{100*bb_done/bb_total:5.1f}%',
})

progress_df = pd.DataFrame(rows)
progress_df


,job,exp,model,seed,done,total,pct
0,exp1l1,1,Llama-3.1-8B-Instruct,42,39,160,24.4%
1,exp1l2,1,Llama-3.1-8B-Instruct,123,39,160,24.4%
2,exp1l3,1,Llama-3.1-8B-Instruct,777,24,160,15.0%
3,exp1g1,1,gemma-3-12b-it,42,24,160,15.0%
4,exp1g2,1,gemma-3-12b-it,123,9,160,5.6%
5,exp1g3,1,gemma-3-12b-it,777,36,160,22.5%
6,exp1q1,1,Qwen3-8B,42,43,160,26.9%
7,exp1q2,1,Qwen3-8B,123,24,160,15.0%
8,exp1q3,1,Qwen3-8B,777,43,160,26.9%
9,exp2l1,2,Llama-3.1-8B-Instruct,42,14,77,18.2%


## Section 2 — Remaining-hours estimate

For each recipe on a given model, take the runtime of the most recent finished run as a proxy for the cost of any future run of that recipe on that model (just a rough estimate — no averaging). Then, for each slurm job, sum the runtimes of recipes that have *not* yet finished for that (model, seed) to get the hours remaining.

In [5]:
# ─── Build per-recipe runtime lookup from finished runs ──────────────────
# runtime_by_recipe[(exp, model, recipe)] = median runtime (seconds) of a single wandb run
# for that (exp, model, recipe) — i.e. ONE (msg, seed) sample, not the full sweep.
from statistics import median

_rt_buckets: dict[tuple[str, str, str], list[float]] = {}
for p in parsed:
    r = p['run']
    rt = r.summary.get('_runtime') or getattr(r, '_attrs', {}).get('runtime')
    if rt is None:
        continue
    _rt_buckets.setdefault((p['exp'], p['model'], p['recipe']), []).append(float(rt))

runtime_by_recipe = {k: median(v) for k, v in _rt_buckets.items()}

# Fallback: average runtime per (exp, recipe) across any model — used when the
# target model hasn't finished even one run of that recipe yet.
_any_buckets: dict[tuple[str, str], list[float]] = {}
for (exp, _model, recipe), rts in _rt_buckets.items():
    _any_buckets.setdefault((exp, recipe), []).extend(rts)
runtime_by_recipe_any_model = {k: median(v) for k, v in _any_buckets.items()}

print(f'per-(exp,model,recipe) entries: {len(runtime_by_recipe)}')
print(f'per-(exp,recipe) fallback:      {len(runtime_by_recipe_any_model)}')


def lookup_runtime(exp: str, model: str, recipe: str) -> tuple[float | None, str]:
    """Return (seconds, source) where source ∈ {'same_model', 'any_model', 'missing'}."""
    v = runtime_by_recipe.get((exp, model, recipe))
    if v is not None:
        return v, 'same_model'
    v = runtime_by_recipe_any_model.get((exp, recipe))
    if v is not None:
        return v, 'any_model'
    return None, 'missing'


def recipes_for(exp: str) -> list[str]:
    return {'1': EXP1_RECIPES, '1bb': EXP1_BB_RECIPES, '2': EXP2_RECIPES}[exp]


def expected_per_recipe(exp: str) -> int:
    # How many wandb runs a *single* recipe contributes to the slurm job.
    if exp == '1':
        return len(MSG_IDS)
    if exp == '1bb':
        return len(MSG_IDS) * len(SEEDS)
    if exp == '2':
        return len(MSG_IDS) + 1   # single (10 msgs) + multi (1)
    raise ValueError(exp)


def done_by_recipe(exp: str, model: str, seed) -> dict[str, int]:
    """Count finished runs per recipe for this (exp, model, seed)."""
    counts: dict[str, int] = {}
    for p in parsed:
        if p['exp'] != exp or p['model'] != model:
            continue
        if seed != 'all' and int(p['seed']) != seed:
            continue
        counts[p['recipe']] = counts.get(p['recipe'], 0) + 1
    return counts


def estimate_remaining_hours(exp: str, model: str, seed) -> tuple[float, int, int]:
    """(hours, n_recipes_using_fallback, n_recipes_no_rt_data)."""
    done = done_by_recipe(exp, model, seed)
    per_recipe_budget = expected_per_recipe(exp)
    total_s = 0.0
    fallback_n = 0
    missing_n = 0
    for recipe in recipes_for(exp):
        remaining = max(0, per_recipe_budget - done.get(recipe, 0))
        if remaining == 0:
            continue
        rt, src = lookup_runtime(exp, model, recipe)
        if rt is None:
            missing_n += 1
            continue
        if src == 'any_model':
            fallback_n += 1
        total_s += rt * remaining
    return total_s / 3600.0, fallback_n, missing_n


est_rows = []
for exp in ['1', '2']:
    for letter, model_full in LETTER_TO_MODEL.items():
        short = MODEL_SHORTS[letter]
        for seed_idx, seed in enumerate(SEEDS, start=1):
            hrs, fb, miss = estimate_remaining_hours(exp, short, seed)
            est_rows.append({
                'job': f'exp{exp}{letter}{seed_idx}',
                'est_hours_left': round(hrs, 2),
                'recipes_fallback_rt': fb,
                'recipes_no_rt': miss,
            })
hrs, fb, miss = estimate_remaining_hours('1bb', BLACKBOX_MODEL_SHORT, 'all')
est_rows.append({
    'job': 'exp1chat',
    'est_hours_left': round(hrs, 2),
    'recipes_fallback_rt': fb,
    'recipes_no_rt': miss,
})
estimate_df = pd.DataFrame(est_rows)
estimate_df


wandb: WARNING A graphql request initiated by the public wandb API timed out (timeout=19 sec). Create a new API with an integer timeout larger than 19, e.g., `api = wandb.Api(timeout=29)` to increase the graphql timeout.


per-(exp,model,recipe) entries: 54
per-(exp,recipe) fallback:      24


,job,est_hours_left,recipes_fallback_rt,recipes_no_rt
0,exp1l1,118.08,0,1
1,exp1l2,118.08,0,1
2,exp1l3,134.09,0,1
3,exp1g1,170.60,0,1
4,exp1g2,190.91,0,1
5,exp1g3,155.98,0,1
6,exp1q1,115.45,0,1
7,exp1q2,132.92,0,1
8,exp1q3,115.45,0,1
9,exp2l1,11.21,0,0


In [6]:
# ─── Combined view ───────────────────────────────────────────────────────
combined = progress_df.merge(
    estimate_df[['job', 'est_hours_left', 'recipes_fallback_rt', 'recipes_no_rt']],
    on='job',
)
combined.sort_values(['exp', 'job']).reset_index(drop=True)

,job,exp,model,seed,done,total,pct,est_hours_left,recipes_fallback_rt,recipes_no_rt
0,exp1g1,1,gemma-3-12b-it,42,24,160,15.0%,170.60,0,1
1,exp1g2,1,gemma-3-12b-it,123,9,160,5.6%,190.91,0,1
2,exp1g3,1,gemma-3-12b-it,777,36,160,22.5%,155.98,0,1
3,exp1l1,1,Llama-3.1-8B-Instruct,42,39,160,24.4%,118.08,0,1
4,exp1l2,1,Llama-3.1-8B-Instruct,123,39,160,24.4%,118.08,0,1
5,exp1l3,1,Llama-3.1-8B-Instruct,777,24,160,15.0%,134.09,0,1
6,exp1q1,1,Qwen3-8B,42,43,160,26.9%,115.45,0,1
7,exp1q2,1,Qwen3-8B,123,24,160,15.0%,132.92,0,1
8,exp1q3,1,Qwen3-8B,777,43,160,26.9%,115.45,0,1
9,exp1chat,1bb,gpt-5-nano,all,0,210,0.0%,19.13,2,5
